In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.sparse import load_npz

# ── Chỉnh tháng snapshot cần EDA ──
MONTH = "2026-04"
GOLD_DIR = f"../data/gold/snapshot={MONTH}"

### 1. Row count và Split ratio

In [2]:
train = pq.read_table(f"{GOLD_DIR}/snapshot=2026-04/full_load/train.parquet").to_pandas()
val   = pq.read_table(f"{GOLD_DIR}/snapshot=2026-04/full_load/val.parquet").to_pandas()
test  = pq.read_table(f"{GOLD_DIR}/snapshot=2026-04/full_load/test.parquet").to_pandas()

total = len(train) + len(val) + len(test)
print(f"Train: {len(train):,} ({len(train)/total:.1%})")
print(f"Val:   {len(val):,}   ({len(val)/total:.1%})")
print(f"Test:  {len(test):,}  ({len(test)/total:.1%})")

Train: 140,274 (80.3%)
Val:   24,755   (14.2%)
Test:  9,726  (5.6%)


### 2. Label balance trong từng split

In [3]:
print(f"Train spam: {train['label'].mean():.1%}")
print(f"Val   spam: {val['label'].mean():.1%}")
print(f"Test  spam: {test['label'].mean():.1%}")
# Kỳ vọng: cả 3 đều gần nhau ~50/50
# Nếu test lệch nhiều → tháng holdout có distribution khác → cần chú ý khi evaluate

Train spam: 46.9%
Val   spam: 46.9%
Test  spam: 45.7%


### 3. Sparse matrix shape và không có NaN

In [4]:
X_train = load_npz(f"{GOLD_DIR}/snapshot=2026-04/full_load/train_X.npz")
X_val   = load_npz(f"{GOLD_DIR}/snapshot=2026-04/full_load/val_X.npz")
X_test  = load_npz(f"{GOLD_DIR}/snapshot=2026-04/full_load/test_X.npz")

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

print(train[["log_chars","avg_word_length","unique_word_ratio","exclaim_count"]].isnull().sum())


X_train: (140274, 30004)
X_val:   (24755, 30004)
X_test:  (9726, 30004)
log_chars            0
avg_word_length      0
unique_word_ratio    0
exclaim_count        0
dtype: int64


### 4. Test set đúng là tháng holdout

In [5]:
# Xác nhận test set được chia đúng tháng chưa
print("Test months:", test["month_partition"].unique())
# Phải là đúng HOLDOUT_MONTH đã config trong gold_build.py
print("Train months:", sorted(train["month_partition"].unique()))

Test months: ['2026-04']
Train months: ['2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03']
